In [0]:
%sql
-- Creating dimension table for patientss
CREATE TABLE IF NOT EXISTS jehangir_catalog.gold.dim_patient
(
Patient_Key STRING,
SRC_PatientID STRING,
FirstName STRING,
LastName STRING,
Patient_Age INTEGER,
datasource STRING
);

insert overwrite jehangir_catalog.gold.dim_patient
select distinct
Patient_Key,
SRC_PatientID,
FirstName,
LastName,
Patient_Age,
datasource 
from jehangir_catalog.silver.patients_silver
where is_quarantined=false;

select * from jehangir_catalog.gold.dim_patient;

In [0]:
# Read data from gold dim_patient table
df_dim_patient = spark.read.table("jehangir_catalog.gold.dim_patient")

# Write to ADLS gold directory as dim_patients
df_dim_patient.write.mode("overwrite").format("delta").save("abfss://gold@jehangirdatalake05.dfs.core.windows.net/gold/dim_patients")

In [0]:
%sql
-- Creating dimension table for Provider.

CREATE TABLE IF NOT EXISTS jehangir_catalog.gold.dim_provider
(
    Provider_Key string,
    provider_id string,
    provider_name string,
    npi_number string,
    dep_id string,
    datasource string
);

insert overwrite jehangir_catalog.gold.dim_provider
select distinct
    concat(provider_id, '-', datasource) AS Provider_Key,
    provider_id,
    provider_name,
    npi_number,
    concat(dep_id,'-',datasource)  deptid,
    datasource 
from jehangir_catalog.silver.providers_silver
where is_quarantined = false;

select * from jehangir_catalog.gold.dim_provider;

In [0]:
# Read data from gold dim_provider table
df_dim_provider = spark.read.table("jehangir_catalog.gold.dim_provider")

# Write to ADLS gold directory as dim_providers
df_dim_provider.write.mode("overwrite").format("delta").save("abfss://gold@jehangirdatalake05.dfs.core.windows.net/gold/dim_providers")

In [0]:
%sql
-- Creating dimension table for Departments.
CREATE TABLE IF NOT EXISTS jehangir_catalog.gold.dim_department
(
Dept_Id string,
SRC_Dept_Id string,
dept_name string,
datasource string
);

insert overwrite jehangir_catalog.gold.dim_department
select 
distinct
Dept_Id ,
SRC_Dept_Id ,
dept_name ,
datasource 
from jehangir_catalog.silver.departments_silver
where is_quarantined=false;

select * from jehangir_catalog.gold.dim_department;

In [0]:
# Read data from gold dim_department table
df_dim_dept = spark.read.table("jehangir_catalog.gold.dim_department")

# Write to ADLS gold directory as dim_departments
df_dim_dept.write.mode("overwrite").format("delta").save("abfss://gold@jehangirdatalake05.dfs.core.windows.net/gold/dim_departments")

In [0]:
%sql
create table if not exists jehangir_catalog.gold.fact_transactions
(
    EncounterID string,
    SRC_EncounterID string,
    FK_Patient_ID string,
    FK_Provider_ID string,
    Encounter_date date,
    Total_amount double,
    Amount_Paid double,
    Amount_Pending double,
    Payment_Status string,
    datasource string,
    Refreshed_at timestamp
);

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Load only changed encounters: silver → gold `fact_transactions`
# MAGIC
# MAGIC **Idea in plain English**
# MAGIC 1. Silver keeps a version number that goes up every time the table changes.
# MAGIC 2. We remember the last silver version we loaded into gold (the *watermark*).
# MAGIC 3. Each run we ask silver: "what changed after that version?" (Delta *Change Data Feed*) and load only those rows.
# MAGIC 4. We save the new version number, but only after the load worked.
# MAGIC
# MAGIC Why not filter on a date column? Your silver merge sets `audit_modifieddate` on every row every run, so a date filter would pick up everything.
# MAGIC
# MAGIC **First run** builds gold from all of silver. **Later runs** only merge the changed rows.
# MAGIC
# MAGIC Only these are touched: `silver.encounters_silver` (read), `gold.fact_transactions` + its ADLS copy (write), and one tiny watermark table.

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 1: Settings

# COMMAND ----------

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE    = "jehangir_catalog.silver.encounters_silver"
GOLD_TABLE      = "jehangir_catalog.gold.fact_transactions"
GOLD_PATH       = "abfss://gold@jehangirdatalake05.dfs.core.windows.net/gold/fact_transactions"
WATERMARK_TABLE = "jehangir_catalog.gold.fact_transactions_watermark"

# Set to True and run the notebook to rebuild gold from scratch, then set it back to False
FULL_REFRESH = True

# The gold columns (what ends up in fact_transactions)
GOLD_COLUMNS = ["EncounterID", "SRC_EncounterID", "FK_Patient_ID", "FK_Provider_ID", "Encounter_date",
                "Total_amount", "Amount_Paid", "Amount_Pending", "Payment_Status", "datasource"]

# Synapse (dedicated pool) may not read Delta deletion vectors, so keep them off for new gold tables
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 2: Create the watermark table (one row: the last silver version loaded)

# COMMAND ----------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
        last_silver_version BIGINT,
        updated_at          TIMESTAMP
    )
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 3: Make sure silver records changes (Change Data Feed)
# MAGIC Switched on once. If it was off, changes made before now are not recorded, so this run does a full load.

# COMMAND ----------

cdf_row = (spark.sql(f"SHOW TBLPROPERTIES {SILVER_TABLE}")
                .filter("key = 'delta.enableChangeDataFeed'")
                .collect())
cdf_was_on = len(cdf_row) > 0 and cdf_row[0]["value"] == "true"

if not cdf_was_on:
    spark.sql(f"ALTER TABLE {SILVER_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print("Change Data Feed switched on for silver encounters")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 4: What do we need to load?

# COMMAND ----------

# Newest version of the silver table
current_version = spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").agg(F.max("version")).first()[0]

# Last version we loaded into gold (None on the very first run)
watermark_row = spark.table(WATERMARK_TABLE).first()
last_version = watermark_row["last_silver_version"] if watermark_row else None

first_run = FULL_REFRESH or (not cdf_was_on) or (last_version is None)

print(f"silver is at version {current_version} | last loaded version: {last_version} | full load: {first_run}")

# Nothing new in silver: stop here
if (not first_run) and last_version >= current_version:
    dbutils.notebook.exit("No new changes in silver. Nothing to load.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 5: Read the changed rows from silver
# MAGIC * First run: the whole silver table as of `current_version`.
# MAGIC * Later runs: only rows inserted, updated or deleted after `last_version`.

# COMMAND ----------

if first_run:
    changes = (spark.sql(f"SELECT * FROM {SILVER_TABLE} VERSION AS OF {current_version}")
                    .withColumn("_change_type", F.lit("insert"))
                    .withColumn("_commit_version", F.lit(current_version).cast("long")))
else:
    changes = (spark.read
                    .option("readChangeFeed", "true")
                    .option("startingVersion", last_version + 1)
                    .option("endingVersion", current_version)
                    .table(SILVER_TABLE)
                    .filter("_change_type IN ('insert', 'update_postimage', 'delete')"))   # ignore the "before" copy of updates

print("Changed rows read from silver:", changes.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 6: Build the gold columns
# MAGIC Same logic as your old `insert into fact_transactions`, plus three helpers:
# MAGIC * keep only the **latest change per EncounterID** (an encounter can change twice between runs)
# MAGIC * `row_hash`: a fingerprint of the row, so unchanged rows are not rewritten
# MAGIC * `is_delete`: true when the row must not be in gold (deleted in silver, or quarantined)

# COMMAND ----------

gold_df = changes.select(
    "EncounterID",
    "SRC_EncounterID",
    F.concat("patient_id", F.lit("-"), "datasource").alias("FK_Patient_ID"),
    F.concat("provider_id", F.lit("-"), "datasource").alias("FK_Provider_ID"),
    F.col("encounter_date").alias("Encounter_date"),
    F.col("total_amount").alias("Total_amount"),
    F.col("amount_paid").alias("Amount_Paid"),
    F.col("amount_pending").alias("Amount_Pending"),
    F.col("payment_status").alias("Payment_Status"),
    "datasource",
    "is_quarantined",
    "_change_type",
    "_commit_version",
)

# Rows without an EncounterID cannot be matched, so skip them
gold_df = gold_df.filter(F.col("EncounterID").isNotNull())

# Keep the newest change for each EncounterID
newest_first = Window.partitionBy("EncounterID").orderBy(F.col("_commit_version").desc())
gold_df = (gold_df.withColumn("rn", F.row_number().over(newest_first))
                  .filter("rn = 1")
                  .drop("rn"))

# Fingerprint of the business columns + flag for rows that must be removed from gold
gold_df = (gold_df
           .withColumn("row_hash", F.sha2(F.to_json(F.struct(*GOLD_COLUMNS)), 256))
           .withColumn("is_delete", (F.col("_change_type") == "delete") | (F.col("is_quarantined") == True)))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 7: Write to gold
# MAGIC * First run: replace the table with everything (this also clears any old duplicates), then copy it to ADLS.
# MAGIC * Later runs: **merge** the changed rows into the table and into the ADLS copy.
# MAGIC   * deleted / quarantined → removed from gold
# MAGIC   * new → inserted
# MAGIC   * existing and really changed (fingerprint differs) → updated

# COMMAND ----------

if first_run:
    full_df = (gold_df.filter("NOT is_delete")
                      .select(*GOLD_COLUMNS, "row_hash")
                      .withColumn("Refreshed_at", F.current_timestamp()))

    # Gold table in Unity Catalog
    (full_df.write.mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(GOLD_TABLE))

    # Copy in ADLS (this was your old "write to ADLS" cell)
    (spark.table(GOLD_TABLE).write.mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(GOLD_PATH))

    print("Full load done")

else:
    # Which columns to set on insert / update
    values = {c: f"s.{c}" for c in GOLD_COLUMNS + ["row_hash"]}
    values["Refreshed_at"] = "current_timestamp()"

    gold_table = DeltaTable.forName(spark, GOLD_TABLE)
    gold_copy_in_adls = DeltaTable.forPath(spark, GOLD_PATH)

    for target in [gold_table, gold_copy_in_adls]:
        (target.alias("t")
               .merge(gold_df.alias("s"), "t.EncounterID = s.EncounterID")
               .whenMatchedDelete(condition="s.is_delete")
               .whenMatchedUpdate(condition="NOT s.is_delete AND t.row_hash <> s.row_hash", set=values)
               .whenNotMatchedInsert(condition="NOT s.is_delete", values=values)
               .execute())

    print("Incremental merge done")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 8: Save the watermark
# MAGIC Only reached if Step 7 worked. If anything above failed, the watermark is unchanged and the next run retries the same changes.

# COMMAND ----------

spark.sql(f"INSERT OVERWRITE {WATERMARK_TABLE} SELECT CAST({current_version} AS BIGINT), current_timestamp()")
print(f"Watermark saved: silver version {current_version}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Step 9: Quick checks

# COMMAND ----------

gold_count = spark.table(GOLD_TABLE).count()
adls_count = spark.read.format("delta").load(GOLD_PATH).count()
duplicates = spark.table(GOLD_TABLE).groupBy("EncounterID").count().filter("count > 1").count()
silver_valid = (spark.table(SILVER_TABLE)
                     .filter("is_quarantined = false AND EncounterID IS NOT NULL")
                     .select("EncounterID").distinct().count())

print(f"gold rows: {gold_count} | ADLS copy rows: {adls_count} | silver valid encounters: {silver_valid} | duplicate keys: {duplicates}")

assert duplicates == 0, "Duplicate EncounterID in gold"
assert gold_count == silver_valid, "Gold and silver row counts differ"
assert adls_count == gold_count, "ADLS copy and gold table differ"

display(spark.table(WATERMARK_TABLE))